# MotionJSON Colab CLI Demo

This notebook is a short CPU/no-model demo. It clones MotionJSON, installs the local package, generates the deterministic red-ball video, runs threshold extraction, validates the output, and creates a ZIP you can download.

Do not use this notebook to host a long-running public web service. Do not paste provider credentials, private videos, SAM2 checkpoints, or API keys into a shared notebook.

In [ ]:
from pathlib import Path
import os
import subprocess

cwd = Path.cwd()
if cwd.name == "json-animated-video" and (cwd / "pyproject.toml").exists():
    repo = cwd
else:
    repo = Path("/content/json-animated-video") if Path("/content").exists() else Path("json-animated-video")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/ptse8204/json-animated-video.git", str(repo)], check=True)
os.chdir(repo)
print(Path.cwd())

In [ ]:
!python3 -m pip install -U pip
!python3 -m pip install -e ".[ui]"

In [ ]:
!python3 -m motionjson.cli backend diagnostics --json

In [ ]:
!python3 examples/make_demo_video.py --out examples/demo_red_ball.mp4
!python3 -m motionjson.cli extract examples/demo_red_ball.mp4 \
  --out out/demo_red_ball \
  --mask-provider threshold \
  --lower-hsv 0,80,80 \
  --upper-hsv 12,255,255 \
  --sample-fps 12 \
  --max-frames 12
!python3 -m motionjson.cli validate out/demo_red_ball

In [ ]:
from pathlib import Path
import json

output = Path("out/demo_red_ball")
for name in [
    "scene_graph.json",
    "object_motion.json",
    "web_asset_manifest.json",
    "tracks.json",
    "fallback_diagnostics.json",
]:
    path = output / name
    print(f"{name}: exists={path.exists()} bytes={path.stat().st_size if path.exists() else 0}")

manifest = json.loads((output / "web_asset_manifest.json").read_text())
print(manifest["schema"])
print(manifest["canvas"])

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

zip_path = Path("motionjson_red_ball_output.zip")
with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for path in Path("out/demo_red_ball").rglob("*"):
        if path.is_file():
            archive.write(path, path.as_posix())
print(zip_path, zip_path.stat().st_size)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print(f"Download helper unavailable outside Colab: {exc}")